In [ ]:
from numpy import hstack
from numpy import zeros
from numpy import ones
from numpy.random import rand
from numpy.random import randn
from numpy.random import randint
from keras.models import Sequential
from keras.layers import Dense
from matplotlib import pyplot

In [ ]:
def generator_model(dim_noise=5, n_output=2):
    model = Sequential()
    model.add(Dense(20, activation='relu',  input_dim=dim_noise))
    model.add(Dense(n_output, activation='linear'))
    return model
    
    

In [ ]:
def discriminator_model(n_input=2):
    model = Sequential()
    model.add(Dense(25,activation='relu',input_dim=n_input))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

In [ ]:
def realdata_generator(n):
    x1 = rand(n)-0.5
    x2 = x1 * x1
    x1 = x1.reshape(n, 1)
    x2 = x2.reshape(n, 1)
    X = hstack((x1 , x2))
    real_lbl=ones((n,1))
    return X,real_lbl

In [ ]:
def generate_noise(dim_noise,n):
    N = randn(dim_noise * n)
    N = N.reshape(n, dim_noise)
    return N

In [ ]:
def fakedata_generator(generator_model, dim_noise, n):
    noise = generate_noise(dim_noise, n)
    fake_sample = generator_model.predict(noise, verbose=0)
    fake_lbl= zeros((n,1))
    return fake_sample , fake_lbl


In [ ]:
def gan(generator, discriminator):
    discriminator.trainable = False
    model= Sequential()

    model.add(generator)

    model.add(discriminator)
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model


In [ ]:
def summarize_performance(epoch, generator, discriminator, latent_dim, n=200):
    # prepare real samples
    x_real, y_real = realdata_generator(n)
    # evaluate discriminator on real examples
    _, acc_real = discriminator.evaluate(x_real, y_real, verbose=0)
    # prepare fake examples
    x_fake, y_fake = fakedata_generator(generator, latent_dim, n)
    # evaluate discriminator on fake examples
    _, acc_fake = discriminator.evaluate(x_fake, y_fake, verbose=0)
    # summarize discriminator performance
    print(epoch, acc_real, acc_fake)
    # scatter plot real and fake data points
    pyplot.scatter(x_real[:, 0], x_real[:, 1], color='red')
    pyplot.scatter(x_fake[:, 0], x_fake[:, 1], color='blue')
    pyplot.show()

In [ ]:
def train(g_model, d_model, gan_model, latent_dim, n_epochs=30000, n_batch=128, n_eval=200):
 # determine half the size of one batch, for updating the discriminator
 half_batch = int(n_batch / 2)
 # manually enumerate epochs
 for i in range(n_epochs):
    # prepare real samples
    x_real, y_real = realdata_generator(half_batch)
    # prepare fake examples
    x_fake, y_fake = fakedata_generator(g_model, latent_dim, half_batch)
    # update discriminator
    d_model.train_on_batch(x_real, y_real)
    d_model.train_on_batch(x_fake, y_fake)
    # prepare points in latent space as input for the generator
    x_gan = generate_noise(latent_dim, n_batch)
    # create inverted labels for the fake samples
    y_gan = ones((n_batch, 1))
    # update the generator via the discriminator's error
    gan_model.train_on_batch(x_gan, y_gan)
    # evaluate the model every n_eval epochs
    if (i+1) % n_eval == 0:
        summarize_performance(i, g_model, d_model, latent_dim)
    
# size of the latent space
latent_dim = 5
# create the discriminator
discriminator = discriminator_model()
# create the generator
generator = generator_model(latent_dim)
# create the gan
gan_model = gan(generator, discriminator)
# train model
train(generator, discriminator, gan_model, latent_dim)